# Fase 2: Preparazione Modeling


**Import & Setup**

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer

pd.set_option('display.max_columns', 500)

In [2]:
df = pd.read_csv("../data/processed/telco_churn_preprocessed.csv")

## 1. Pulizia Finale Dataset


### 1.1 Rimozione Colonne Non Necessarie


Rimuoviamo le colonne identificate come non utili per il modeling:
- customerID: identificativo univoco, non ha potere predittivo
- TotalCharges: ridondante con tenure (VIF = 10.4, correlazione 0.83)
- Churn: manteniamo solo la versione numerica (Numeric_Churn)

In [3]:
df = df.drop(["customerID","TotalCharges", "Churn"], axis=1)


### 1.2 Encoding Variabili Residue

La colonna tenure_group è ancora in formato stringa. 
Applichiamo l'encoding appropriato.

In [4]:

ordine_tenure = ["0-12", "13-24", "25-48", "49-72"]

enc = OrdinalEncoder(categories=[ordine_tenure])

df["tenure_group"] = enc.fit_transform(df[["tenure_group"]])



### 1.3 Verifica Finale

Controlliamo che tutte le colonne siano numeriche e pronte per il modeling.


In [5]:
print(df.dtypes)

gender                                       int64
SeniorCitizen                                int64
Partner                                      int64
Dependents                                   int64
tenure                                       int64
PhoneService                                 int64
MultipleLines                                int64
OnlineSecurity                               int64
OnlineBackup                                 int64
DeviceProtection                             int64
TechSupport                                  int64
StreamingTV                                  int64
StreamingMovies                              int64
Contract                                     int64
PaperlessBilling                             int64
MonthlyCharges                             float64
Churn_numeric                                int64
InternetService_DSL                           bool
InternetService_Fiber optic                   bool
InternetService_No             

## 2. Train/Test Split


### 2.1 Separazione Feature e Target


In [6]:
X  = df.drop(columns=["Churn_numeric"])
y = df["Churn_numeric"]



Separiamo le feature (X) dalla variabile target (y).


### 2.2 Split Stratificato


Dividiamo il dataset in training (80%) e test (20%) set.
Utilizziamo la stratificazione per mantenere la proporzione delle classi 
in entrambi i set, dato lo sbilanciamento identificato nell'EDA.

In [7]:
X_train, X_test, y_train, y_test = train_test_split (X, y, 
                                                test_size=0.2, 
                                                random_state=42, 
                                                stratify=y
                                                )


### 2.3 Verifica Split


Controlliamo le dimensioni e la distribuzione delle classi in entrambi i set.


In [8]:
print(f"Training set: {X_train.shape[0]}")
print(f"Testing set: {X_test.shape[0]}")

print(f"Proporzione Training set {y_train.value_counts(normalize=True)}")
print(f"Proporzione Testing set {y_test.value_counts(normalize=True)}")

Training set: 5634
Testing set: 1409
Proporzione Training set Churn_numeric
0    0.734647
1    0.265353
Name: proportion, dtype: float64
Proporzione Testing set Churn_numeric
0    0.734564
1    0.265436
Name: proportion, dtype: float64


Output atteso: 
- Training set: circa 5634 campioni
- Test set: circa 1409 campioni
- Proporzione classi mantenuta in entrambi

## 3. Scaling Feature Numeriche


### 3.1 Identificazione Feature da Scalare


Lo scaling è necessario per le feature continue con range ampi.
Le feature binarie e categoriche encoded non richiedono scaling.

Feature da scalare:
- [MonthlyCharges, tenure]

In [9]:
numeric_features = X.select_dtypes(include=["number"])

continous_features = [col for col in numeric_features.columns if X[col].nunique() > 20 ]

print(continous_features)

['tenure', 'MonthlyCharges']


### 3.2 Scelta dello Scaler

Ho scelto StandardScaler perché il dataset non presenta outlier e poiché c'è una distrbuzione nelle feature abbastanza normale.

In [10]:
colmn_to_scale = ["MonthlyCharges", "tenure"]

preprocessor = ColumnTransformer(
    transformers =[
        ("scaler", StandardScaler(), colmn_to_scale)],
        remainder="passthrough"
)



### 3.3 Applicazione Scaling

Applichiamo lo scaler con fit solo sul training set per evitare data leakage.

In [11]:
X_train_scaled = preprocessor.fit_transform(X_train).astype(np.float64)
X_test_scaled = preprocessor.transform(X_test).astype(np.float64)

### 3.4 Verifica Scaling

Controlliamo le statistiche delle feature scalate.


In [21]:
means = np.mean(X_train_scaled, axis=0)
stds = np.std(X_train_scaled, axis=0)

nuove_colonne = preprocessor.get_feature_names_out()

pd_train_scaled = pd.DataFrame(X_train_scaled, columns=nuove_colonne).astype(np.float64)
pd_test_scaled = pd.DataFrame(X_test_scaled, columns=nuove_colonne).astype(np.float64)


print("Caratteristiche feature scalate nel training:\n", pd_train_scaled[["scaler__MonthlyCharges", "scaler__tenure"]].describe().loc[["mean","std"]])
print("Caratteristiche feature scalate nel test:\n", pd_test_scaled[["scaler__MonthlyCharges", "scaler__tenure"]].describe().loc[["mean","std"]])




Caratteristiche feature scalate nel training:
       scaler__MonthlyCharges  scaler__tenure
mean           -2.402527e-16   -1.008935e-17
std             1.000089e+00    1.000089e+00
Caratteristiche feature scalate nel test:
       scaler__MonthlyCharges  scaler__tenure
mean               -0.027911       -0.023184
std                 0.992131        0.998342


**Nota**: Il test set avrà statistiche leggermente diverse perché 
il fit è stato fatto solo sul training set.